# Chart widget

Browse Yahoo Finance prices with centered Ticker and Max bars controls. The widget handles the input layout and chart updates; the notebook supplies a price loader.

Install the optional notebook dependencies and Yahoo client in your notebook environment:

```console
pip install 'mplchart[notebook,pandas]' yfinance
```

Run this notebook with a live kernel and widget support to use the controls. Fetching Yahoo data requires internet access; a static preview cannot run the loader.

## Load Yahoo prices

`chart_widget` accepts any callable that takes a ticker and returns a pandas or Polars prices DataFrame. Here `yfinance` supplies pandas data, and `normalize_prices` prepares its columns and date index for mplchart.

Fetch five years so moving averages have history before the visible window. The small in-memory cache avoids another Yahoo request when revisiting a ticker. Run `get_prices.cache_clear()` before creating a new widget to refresh cached prices.

In [3]:
from functools import lru_cache

import pandas as pd
import yfinance as yf

from mplchart.notebook import chart_widget
from mplchart.utils import normalize_prices


@lru_cache(maxsize=32)
def get_prices(ticker: str, period: str = "5y") -> pd.DataFrame:
    prices = yf.Ticker(ticker).history(period=period, auto_adjust=True)
    if prices.empty:
        raise ValueError(f"No Yahoo prices found for {ticker!r}")
    return normalize_prices(prices)

## Browse a chart

Enter a ticker such as `AAPL`, `MSFT`, or `SPY`, then press Enter or leave the field to update the chart. Max bars controls only the visible window and reuses the loaded history.

The simplest call is `chart_widget(get_prices)`, which shows candlesticks and volume. This example supplies a full plot sequence with two moving averages and an RSI pane. Leave the widget as the last expression in the cell to display it.

In [4]:
from mplchart.indicators import SMA, RSI
from mplchart.primitives import Candlesticks, Volume, Pane, Line

indicators = [
    Candlesticks(),
    SMA(50),
    SMA(200),
    Volume(),
    Pane("below", yticks=(30, 50, 70)),
    Line(RSI(14), overbought=70, oversold=30),
]

chart_widget(get_prices, ticker="AAPL", max_bars=250, indicators=indicators)

## Use another loader

For an existing bardata feed, pass `feed.get` instead of `get_prices`. To bind query options, use `functools.partial`, for example `partial(feed.get, freq="weekly")`. Use indicators from `mplchart.expressions` if your loader returns Polars data.

See the [notebook API reference](../reference/notebook.md) for chart options and loader behavior.